**Lab type:** debug

**Course:** ML101 — Intro to Machine Learning

**Lesson:** Classification Fundamentals — Predicting Categories

**Task:** The AI-generated analysis below contains 3 bugs. For each bug: identify what is wrong, explain why the output is misleading, and write the corrected code in the fix cell.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

np.random.seed(42)
n = 1000
# Imbalanced: 95% not churned, 5% churned
churned = np.zeros(n)
churned[:50] = 1  # 5% positive
np.random.shuffle(churned)

account_age = np.where(churned == 1,
    np.random.normal(8, 3, n),
    np.random.normal(24, 6, n)
).clip(1, 60)
monthly_spend = np.where(churned == 1,
    np.random.normal(30, 15, n),
    np.random.normal(80, 20, n)
).clip(5, 200)
support_tickets = np.where(churned == 1,
    np.random.poisson(5, n),
    np.random.poisson(1, n)
)

df = pd.DataFrame({
    'account_age': account_age,
    'monthly_spend': monthly_spend,
    'support_tickets': support_tickets,
    'churned': churned.astype(int)
})
print(f"Dataset shape: {df.shape}")
print(f"Class balance:\n{df['churned'].value_counts()}")

## Step 1: Initial Model Performance

We train a logistic regression model to predict customer churn and report its performance on the held-out test set.

In [ ]:
# AI-generated — initial churn model training and evaluation
X = df[['account_age', 'monthly_spend', 'support_tickets']]
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

model = LogisticRegression(random_state=42)
model.fit(X_train_sc, y_train)
y_pred = model.predict(X_test_sc)

acc = accuracy_score(y_test, y_pred)
print(f"Model accuracy: {acc:.3f}")  # <- Bug 1
print("Model is performing well!")

**Bug 1 Investigation:** Run the cell above. The accuracy looks high — over 90%. But look at the class balance printed in Setup. What would a naive classifier that always predicts 'not churned' achieve? Is accuracy the right metric to report here?

In [ ]:
# Fix Bug 1 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 1</summary>

**What was wrong:** The code reported accuracy as the headline metric on a dataset with 95% negative class. A dummy classifier that always predicts "not churned" would achieve ~95% accuracy without learning anything — so a high accuracy score here is meaningless.

**Why it matters:** Accuracy inflates apparent model quality on imbalanced datasets. It hides the fact that the model may be missing nearly all of the churners (the class we actually care about).

**Correct approach:** Report recall (what fraction of actual churners were caught), precision (what fraction of flagged customers actually churned), and F1 (their harmonic mean). These metrics expose whether the model is genuinely useful for the minority class.

</details>

## Step 2: Adjusting the Decision Threshold

The business team wants to run a proactive retention campaign. Missing a churner is expensive — the cost of a lost customer far exceeds the cost of an unnecessary retention offer. We now look at the model's probability outputs and decide on a classification threshold.

In [ ]:
# AI-generated — applying a classification threshold to the predicted probabilities
proba = model.predict_proba(X_test_sc)[:, 1]
y_pred_default = (proba >= 0.5).astype(int)  # <- Bug 2
recall = recall_score(y_test, y_pred_default)
precision = precision_score(y_test, y_pred_default)
print(f"Threshold: 0.5")
print(f"Recall:    {recall:.3f}")
print(f"Precision: {precision:.3f}")
print("This threshold is appropriate for catching churners.")

**Bug 2 Investigation:** Run the cell above. Given the business context described above (missing a churner is expensive), does a threshold of 0.5 make sense? What happens to recall if you lower the threshold? Is the comment at the bottom of the cell justified?

In [ ]:
# Fix Bug 2 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 2</summary>

**What was wrong:** The threshold of 0.5 was used without considering the asymmetric cost of false negatives. When missing a churner is far more expensive than a wasted retention offer, you want higher recall even at the cost of lower precision.

**Why it matters:** At threshold 0.5, the model may classify most churners as "safe" (false negatives) because churner probability scores are spread across the range. Lowering the threshold flags more customers for follow-up, catching more real churners.

**Correct approach:** Sweep thresholds (e.g. 0.2, 0.3, 0.4) and compare recall/precision trade-offs. Choose the threshold aligned with the business cost structure — here, a threshold around 0.2–0.3 maximises recall while keeping precision workable.

</details>

## Step 3: Re-scaling for a Revised Pipeline

A colleague suggests trying a revised feature set and rebuilds the preprocessing pipeline from scratch.

In [ ]:
# AI-generated — revised preprocessing pipeline
X2 = df[['account_age', 'monthly_spend', 'support_tickets']]
y2 = df['churned']

scaler2 = StandardScaler()
X2_scaled = scaler2.fit_transform(X2)  # <- Bug 3

X_tr, X_te, y_tr, y_te = train_test_split(X2_scaled, y2, test_size=0.2, random_state=42)

model2 = LogisticRegression(random_state=42)
model2.fit(X_tr, y_tr)
print(f"Revised model F1: {f1_score(y_te, model2.predict(X_te)):.3f}")

**Bug 3 Investigation:** Run the cell above. The code runs fine and produces a result. Compare the order of operations here to the Setup cell in Step 1 (which was done correctly). What information has the scaler seen that it should not have?

In [ ]:
# Fix Bug 3 here
# YOUR CODE HERE

**Explanation:** Write your answer here — what was wrong and why the fix is correct.

<details>
<summary>🔑 Reveal answer — Bug 3</summary>

**What was wrong:** `scaler2.fit_transform(X2)` was called on the full dataset before `train_test_split`. The scaler computed mean and standard deviation from all rows, including future test rows — a data leakage violation.

**Why it matters:** Even though the numerical impact on a large dataset can be small, this pipeline does not reflect production conditions: in deployment, the scaler will only have seen training data. Metrics computed this way are subtly optimistic and the pipeline will not behave identically on new data.

**Correct approach:** Call `train_test_split` first, then `scaler.fit_transform(X_train)` and `scaler.transform(X_test)` — exactly as done correctly in Step 1.

</details>

## Corrected Analysis

The cell below applies all three fixes in the correct order. Run it end-to-end to confirm the pipeline is now sound.

In [ ]:
# --- Corrected pipeline: all three bugs fixed ---

X = df[['account_age', 'monthly_spend', 'support_tickets']]
y = df['churned']

# Fix 3: split BEFORE fitting the scaler
X_train_raw, X_test_raw, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_fixed = StandardScaler()
X_train_sc = scaler_fixed.fit_transform(X_train_raw)
X_test_sc = scaler_fixed.transform(X_test_raw)

model_fixed = LogisticRegression(random_state=42)
model_fixed.fit(X_train_sc, y_train)
y_pred = model_fixed.predict(X_test_sc)

# Fix 1: use F1 and recall instead of accuracy for imbalanced classes
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
print("Performance at default threshold (0.5):")
print(f"  Accuracy:  {acc:.3f}  <- inflated by class imbalance")
print(f"  Recall:    {rec:.3f}  <- fraction of actual churners caught")
print(f"  Precision: {prec:.3f}")
print(f"  F1:        {f1:.3f}")

# Fix 2: explore multiple thresholds to optimise for recall
proba = model_fixed.predict_proba(X_test_sc)[:, 1]
print("\nThreshold analysis (lower threshold -> higher recall):")
print(f"{'Threshold':>10}  {'Recall':>8}  {'Precision':>10}  {'F1':>8}")
for threshold in [0.2, 0.3, 0.4, 0.5]:
    y_t = (proba >= threshold).astype(int)
    r = recall_score(y_test, y_t, zero_division=0)
    p = precision_score(y_test, y_t, zero_division=0)
    f = f1_score(y_test, y_t, zero_division=0)
    print(f"{threshold:>10.1f}  {r:>8.3f}  {p:>10.3f}  {f:>8.3f}")
print("\nChoose the threshold that best balances recall against precision for your business context.")

<details>
<summary>🔑 Reveal summary answers</summary>

1. **Use class-appropriate metrics:** On imbalanced datasets, accuracy is misleading — report recall, precision, and F1 to measure performance on the minority class.

2. **Choose the threshold for your cost structure:** A 0.5 threshold is rarely optimal for imbalanced problems; lower it when false negatives are expensive.

3. **Fit preprocessors after splitting:** Fitting the scaler on the full dataset before the split is data leakage — the scaler must only see training-fold statistics.

</details>